In [1]:
import os
import sys
import json
import glob
import gc

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType
import warnings 
warnings.filterwarnings('ignore')

In [2]:
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [9]:
df = session.sql("""
    SELECT PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
    WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN (
                 SELECT *
                 FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)
    AND CAL_DATE BETWEEN '2023-09-01' AND '2023-12-10'
    UNION ALL
    SELECT PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
    WHERE CAL_DATE BETWEEN '2024-09-01' AND '2024-12-10'
    AND PARENT_DEALER_CODE_MODEL_FAMILY IN (
                 SELECT *
                 FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)
    UNION ALL
    SELECT PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
    WHERE CAL_DATE BETWEEN '2025-09-01' AND '2025-12-10'
        AND PARENT_DEALER_CODE_MODEL_FAMILY IN (
                 SELECT *
                 FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)
    UNION ALL
    SELECT PARENT_DEALER_CODE_MODEL_FAMILY,
                 CAL_DATE,PRED_MEAN AS NET_SALES
    FROM MOP_DATABASE.SOQ.TFT_PREDICTIONS_ITERATION3_MEAN
    WHERE CAL_DATE>='2026-09-01'
    """)